In [0]:
from pyspark.sql.functions import (
    col,
    sum,
    count,
    avg
)

orders_silver_df = spark.table(
    "workspace.zomato_silver.orders"
)

daily_sales_df = (
    orders_silver_df
    .groupBy("order_date")
    .agg(
        sum("sales_qty").alias("total_sales_qty"),
        sum("sales_amount").alias("total_sales_amount"),
        count("*").alias("total_orders"),
        avg("sales_amount").alias("avg_order_amount")
    )
    .orderBy("order_date")
)

display(daily_sales_df)

order_date,total_sales_qty,total_sales_amount,total_orders,avg_order_amount
2017-10-06,1,505.0,1,505.0
2017-10-09,1,185.0,1,185.0
2017-10-10,340,184801.0,2,92400.5
2017-10-11,7,6204.0,1,6204.0
2017-10-13,2,2070.0,2,1035.0
2017-10-16,1,495.0,1,495.0
2017-10-20,1,2023.0,1,2023.0
2017-10-23,20,25130.0,1,25130.0
2017-10-24,1,380.0,1,380.0
2017-10-25,2,1546.0,2,773.0


In [0]:
print(
    "Daily sales rows:",
    daily_sales_df.count()
)

print(
    "Distinct order dates:",
    orders_silver_df
    .select("order_date")
    .distinct()
    .count()
)

Daily sales rows: 209
Distinct order dates: 209


In [0]:
daily_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.zomato_gold.daily_sales"
    )

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.zomato_gold.daily_sales
        ORDER BY order_date
    """)
)

order_date,total_sales_qty,total_sales_amount,total_orders,avg_order_amount
2017-10-06,1,505.0,1,505.0
2017-10-09,1,185.0,1,185.0
2017-10-10,340,184801.0,2,92400.5
2017-10-11,7,6204.0,1,6204.0
2017-10-13,2,2070.0,2,1035.0
2017-10-16,1,495.0,1,495.0
2017-10-20,1,2023.0,1,2023.0
2017-10-23,20,25130.0,1,25130.0
2017-10-24,1,380.0,1,380.0
2017-10-25,2,1546.0,2,773.0


In [0]:
from pyspark.sql.functions import (
    col,
    sum,
    count,
    avg,
    max,
    min
)

orders_df = spark.table(
    "workspace.zomato_silver.orders"
)

restaurants_df = spark.table(
    "workspace.zomato_gold.dim_restaurants"
)

restaurant_performance_df = (
    orders_df
    .join(
        restaurants_df,
        orders_df.r_id == restaurants_df.restaurant_id,
        "inner"
    )
    .groupBy(
        restaurants_df.restaurant_id,
        restaurants_df.name,
        restaurants_df.city,
        restaurants_df.rating,
        restaurants_df.cuisine
    )
    .agg(
        count("*").alias("total_orders"),
        sum("sales_qty").alias("total_sales_qty"),
        sum("sales_amount").alias("total_sales_amount"),
        avg("sales_amount").alias("avg_order_amount"),
        max("sales_amount").alias("max_order_amount"),
        min("sales_amount").alias("min_order_amount")
    )
)

display(
    restaurant_performance_df
    .orderBy(col("total_sales_amount").desc())
    .limit(20)
)

restaurant_id,name,city,rating,cuisine,total_orders,total_sales_qty,total_sales_amount,avg_order_amount,max_order_amount,min_order_amount
158193,yummy hub,Abohar,null,Indian,1,310,170185.0,170185.0,170185.0,170185.0
385636,CAKE PLANET,Adityapur,null,"Bakery,Desserts",1,240,143560.0,143560.0,143560.0,143560.0
158195,wah ji waah veg and non veg corner,Abohar,null,"North Indian,Chinese",1,93,126296.0,126296.0,126296.0,126296.0
257181,Shri Balaji fast food and Variety store,Abohar,null,Indian,1,79,107500.0,107500.0,107500.0,107500.0
460692,Verma Dhaba,Abohar,null,Indian,1,90,105301.0,105301.0,105301.0,105301.0
407249,CHAWLA SAAB THE JUICE MASTER,Abohar,null,"Juices,Beverages",1,184,101194.0,101194.0,101194.0,101194.0
427610,Just Baked,Abohar,null,"Beverages,Pizzas",1,81,76329.0,76329.0,76329.0,76329.0
397980,Chaw Box,Adityapur,3.3,"Chinese,North Indian",1,200,66111.0,66111.0,66111.0,66111.0
394554,Roll Express,Abohar,4.0,Fast Food,1,38,52319.0,52319.0,52319.0,52319.0
567335,AB FOODS POINT,Abohar,null,"Beverages,Pizzas",1,100,41241.0,41241.0,41241.0,41241.0


In [0]:
print(
    "Restaurant performance rows:",
    restaurant_performance_df.count()
)

print(
    "Distinct restaurants:",
    restaurant_performance_df
    .select("restaurant_id")
    .distinct()
    .count()
)

Restaurant performance rows: 281
Distinct restaurants: 281


In [0]:
restaurant_performance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.zomato_gold.restaurant_performance"
    )

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.zomato_gold.restaurant_performance
        ORDER BY total_sales_amount DESC
        LIMIT 20
    """)
)

restaurant_id,name,city,rating,cuisine,total_orders,total_sales_qty,total_sales_amount,avg_order_amount,max_order_amount,min_order_amount
158193,yummy hub,Abohar,null,Indian,1,310,170185.0,170185.0,170185.0,170185.0
385636,CAKE PLANET,Adityapur,null,"Bakery,Desserts",1,240,143560.0,143560.0,143560.0,143560.0
158195,wah ji waah veg and non veg corner,Abohar,null,"North Indian,Chinese",1,93,126296.0,126296.0,126296.0,126296.0
257181,Shri Balaji fast food and Variety store,Abohar,null,Indian,1,79,107500.0,107500.0,107500.0,107500.0
460692,Verma Dhaba,Abohar,null,Indian,1,90,105301.0,105301.0,105301.0,105301.0
407249,CHAWLA SAAB THE JUICE MASTER,Abohar,null,"Juices,Beverages",1,184,101194.0,101194.0,101194.0,101194.0
427610,Just Baked,Abohar,null,"Beverages,Pizzas",1,81,76329.0,76329.0,76329.0,76329.0
397980,Chaw Box,Adityapur,3.3,"Chinese,North Indian",1,200,66111.0,66111.0,66111.0,66111.0
394554,Roll Express,Abohar,4.0,Fast Food,1,38,52319.0,52319.0,52319.0,52319.0
567335,AB FOODS POINT,Abohar,null,"Beverages,Pizzas",1,100,41241.0,41241.0,41241.0,41241.0


In [0]:
from pyspark.sql.functions import (
    col,
    sum,
    count,
    avg,
    max,
    min
)

orders_df = spark.table(
    "workspace.zomato_silver.orders"
)

users_df = spark.table(
    "workspace.zomato_gold.dim_users"
)

customer_order_analysis_df = (
    orders_df
    .join(
        users_df,
        "user_id",
        "inner"
    )
    .groupBy(
        "user_id",
        "name",
        "age",
        "gender",
        "marital_status",
        "occupation",
        "monthly_income",
        "educational_qualifications",
        "family_size"
    )
    .agg(
        count("*").alias("total_orders"),
        sum("sales_qty").alias("total_sales_qty"),
        sum("sales_amount").alias("total_sales_amount"),
        avg("sales_amount").alias("avg_order_amount"),
        max("sales_amount").alias("max_order_amount"),
        min("sales_amount").alias("min_order_amount")
    )
)

display(
    customer_order_analysis_df
    .orderBy(col("total_sales_amount").desc())
    .limit(20)
)

user_id,name,age,gender,marital_status,occupation,monthly_income,educational_qualifications,family_size,total_orders,total_sales_qty,total_sales_amount,avg_order_amount,max_order_amount,min_order_amount
72391,Katrina Vazquez,26,Male,Married,Employee,10001 to 25000,Graduate,4,1,310,170185.0,170185.0,170185.0,170185.0
82777,Anthony Baker,23,Male,Single,Student,No Income,Post Graduate,2,1,240,143560.0,143560.0,143560.0,143560.0
29210,Samuel Ward,18,Male,Single,Student,No Income,Graduate,5,1,93,126296.0,126296.0,126296.0,126296.0
44025,Julie Gomez,29,Female,Married,Employee,More than 50000,Graduate,3,1,79,107500.0,107500.0,107500.0,107500.0
80146,David Carlson,26,Male,Single,Employee,10001 to 25000,Graduate,2,1,90,105301.0,105301.0,105301.0,105301.0
91457,Debbie Leonard,24,Male,Married,Employee,More than 50000,Post Graduate,3,1,184,101194.0,101194.0,101194.0,101194.0
29312,Michelle Stevens,25,Male,Single,Self Employeed,10001 to 25000,Graduate,3,1,81,76329.0,76329.0,76329.0,76329.0
60230,Mark Hernandez,28,Male,Married,Self Employeed,10001 to 25000,Graduate,2,1,200,66111.0,66111.0,66111.0,66111.0
29585,Stephen Wheeler,23,Female,Single,Student,No Income,Graduate,5,1,38,52319.0,52319.0,52319.0,52319.0
49226,Teresa Garcia,27,Male,Married,Self Employeed,25001 to 50000,Graduate,6,1,100,41241.0,41241.0,41241.0,41241.0


In [0]:
print(
    "Customer analysis rows:",
    customer_order_analysis_df.count()
)

print(
    "Distinct customers:",
    customer_order_analysis_df
    .select("user_id")
    .distinct()
    .count()
)

Customer analysis rows: 281
Distinct customers: 281


In [0]:
customer_order_analysis_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.zomato_gold.customer_order_analysis"
    )

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.zomato_gold.customer_order_analysis
        ORDER BY total_sales_amount DESC
        LIMIT 20
    """)
)

user_id,name,age,gender,marital_status,occupation,monthly_income,educational_qualifications,family_size,total_orders,total_sales_qty,total_sales_amount,avg_order_amount,max_order_amount,min_order_amount
72391,Katrina Vazquez,26,Male,Married,Employee,10001 to 25000,Graduate,4,1,310,170185.0,170185.0,170185.0,170185.0
82777,Anthony Baker,23,Male,Single,Student,No Income,Post Graduate,2,1,240,143560.0,143560.0,143560.0,143560.0
29210,Samuel Ward,18,Male,Single,Student,No Income,Graduate,5,1,93,126296.0,126296.0,126296.0,126296.0
44025,Julie Gomez,29,Female,Married,Employee,More than 50000,Graduate,3,1,79,107500.0,107500.0,107500.0,107500.0
80146,David Carlson,26,Male,Single,Employee,10001 to 25000,Graduate,2,1,90,105301.0,105301.0,105301.0,105301.0
91457,Debbie Leonard,24,Male,Married,Employee,More than 50000,Post Graduate,3,1,184,101194.0,101194.0,101194.0,101194.0
29312,Michelle Stevens,25,Male,Single,Self Employeed,10001 to 25000,Graduate,3,1,81,76329.0,76329.0,76329.0,76329.0
60230,Mark Hernandez,28,Male,Married,Self Employeed,10001 to 25000,Graduate,2,1,200,66111.0,66111.0,66111.0,66111.0
29585,Stephen Wheeler,23,Female,Single,Student,No Income,Graduate,5,1,38,52319.0,52319.0,52319.0,52319.0
49226,Teresa Garcia,27,Male,Married,Self Employeed,25001 to 50000,Graduate,6,1,100,41241.0,41241.0,41241.0,41241.0


In [0]:
from pyspark.sql.functions import col, regexp_extract

menus_df = (
    spark.table("workspace.zomato_silver.menus")
    .alias("m")
)

foods_df = (
    spark.table("workspace.zomato_gold.dim_foods")
    .alias("f")
)

restaurants_df = (
    spark.table("workspace.zomato_gold.dim_restaurants")
    .alias("r")
)

food_menu_analysis_df = (
    menus_df
    .join(
        foods_df,
        col("m.f_id") == col("f.food_id"),
        "left"
    )
    .join(
        restaurants_df,
        col("m.r_id").cast("long") == col("r.restaurant_id"),
        "left"
    )
    .select(
        col("m.menu_id").alias("menu_id"),
        col("m.r_id").cast("long").alias("restaurant_id"),
        col("r.name").alias("restaurant_name"),
        col("r.city").alias("city"),
        col("f.food_id").alias("food_id"),
        col("f.food_name").alias("food_name"),
        col("f.food_type").alias("food_type"),
        col("m.cuisine").alias("cuisine"),
        regexp_extract(
            col("m.price"),
            r"(\d+(?:\.\d+)?)",
            1
        ).cast("double").alias("menu_price")
    )
)

display(
    food_menu_analysis_df.limit(20)
)

menu_id,restaurant_id,restaurant_name,city,food_id,food_name,food_type,cuisine,menu_price
mn612843,435679,LunchBox - Meals and Thalis,Amritsar,fd83726,Double Omelette with Masala Bread,Non-veg,"North Indian,Chinese",139.0
mn917348,477214,Food Court Biryani Express,"JP Nagar,Bangalore",fd42879,Egg Fried Rice,Veg,"Biryani,Snacks",60.0
mn635182,216110,Shree Sai Nath Restaurant,Anand,fd817,Veg Biryani,Veg,North Indian,109.0
mn1047619,302902,HSR Food Point,"HSR,Bangalore",fd56113,Egg Curry (2 Eggs),Non-veg,"North Indian,Chinese",99.0
mn374972,492236,Momo Guy,"Maninagar,Ahmedabad",fd155931,Veg Schezwan Momos - 6 Pcs,Veg,"Tibetan,Asian",209.0
mn528883,486412,Out Town Cafe,Alwar,fd76798,Seasonal Vegetable,Veg,"Snacks,Chinese",249.0
mn12610,174458,Milanee s Kitchen,Adityapur,fd47104,Veg Manchow Soup,Non-veg,"Bengali,Indian",110.0
mn243906,529533,Magic Pancakes,"Bopal,Ahmedabad",fd100864,Waffle Ferrero Rocher Milkshake,Veg,"Waffle,Desserts",129.0
mn850398,218,Anand Sweets and Savouries,"Koramangala,Bangalore",fd65709,Chandrakala,Veg,"Sweets,Snacks",180.95
mn164627,63478,Shivam Snacks,"GOTA,Ahmedabad",fd164060,Special Lassi (300 ml),Veg,"Snacks,Indian",110.0


In [0]:
print(
    "Food menu analysis rows:",
    food_menu_analysis_df.count()
)

Food menu analysis rows: 665517


In [0]:
print(
    "Menus without restaurant:",
    food_menu_analysis_df
    .filter(
        col("restaurant_id").isNotNull() &
        col("restaurant_name").isNull()
    )
    .count()
)

Menus without restaurant: 160


In [0]:
print(
    "Menus without food:",
    food_menu_analysis_df
    .filter(col("food_id").isNull())
    .count()
)

Menus without food: 0


In [0]:
food_menu_analysis_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.zomato_gold.food_menu_analysis"
    )

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.zomato_gold.food_menu_analysis
        LIMIT 20
    """)
)

menu_id,restaurant_id,restaurant_name,city,food_id,food_name,food_type,cuisine,menu_price
mn612843,435679,LunchBox - Meals and Thalis,Amritsar,fd83726,Double Omelette with Masala Bread,Non-veg,"North Indian,Chinese",139.0
mn917348,477214,Food Court Biryani Express,"JP Nagar,Bangalore",fd42879,Egg Fried Rice,Veg,"Biryani,Snacks",60.0
mn635182,216110,Shree Sai Nath Restaurant,Anand,fd817,Veg Biryani,Veg,North Indian,109.0
mn1047619,302902,HSR Food Point,"HSR,Bangalore",fd56113,Egg Curry (2 Eggs),Non-veg,"North Indian,Chinese",99.0
mn374972,492236,Momo Guy,"Maninagar,Ahmedabad",fd155931,Veg Schezwan Momos - 6 Pcs,Veg,"Tibetan,Asian",209.0
mn528883,486412,Out Town Cafe,Alwar,fd76798,Seasonal Vegetable,Veg,"Snacks,Chinese",249.0
mn12610,174458,Milanee s Kitchen,Adityapur,fd47104,Veg Manchow Soup,Non-veg,"Bengali,Indian",110.0
mn243906,529533,Magic Pancakes,"Bopal,Ahmedabad",fd100864,Waffle Ferrero Rocher Milkshake,Veg,"Waffle,Desserts",129.0
mn850398,218,Anand Sweets and Savouries,"Koramangala,Bangalore",fd65709,Chandrakala,Veg,"Sweets,Snacks",180.95
mn164627,63478,Shivam Snacks,"GOTA,Ahmedabad",fd164060,Special Lassi (300 ml),Veg,"Snacks,Indian",110.0


In [0]:
print(
    "Gold food_menu_analysis rows:",
    spark.table(
        "workspace.zomato_gold.food_menu_analysis"
    ).count()
)

Gold food_menu_analysis rows: 665517
